# DS 3001 Semester Project (Mid-Semester Checkpoint)
## Classifying Crime Type in DC: Predicting Violent vs. Property Crime

**Team ID:** Team 13  
**Team Members:** Sufyan Billel, Ryan Fang, Eloise McDonough  
**Status:** Halfway-Point Milestone (Data Wrangling, Exploratory Data Analysis, and Preliminary Baseline Models)

---

### 1. Question, Scope, and Data Provenance

* **Research Question:** Using DC's crime incident reports, do time of day, method, and location predict whether an incident is violent or property crime, and where do these results concentrate?
* **Data Source & Provenance:** DC Metropolitan Police Department (MPD) "Crime Incidents" reports (2024–2025) accessed via Open Data DC (`https://opendata.dc.gov/`). The dataset is public, documented, and directly accessible without licensing barriers.
* **Unit of Observation:** A single reported crime incident in Washington, DC.
* **Analytic Sample Criteria:**
  * **Included Data:** DC MPD Crime Incident reports from 2024–2025.
  * **Excluded Data:** Incidents missing offense classifications or geographic coordinates (`LATITUDE`, `LONGITUDE`), as well as 311 service requests or non-MPD records.
* **Key Variables:**
  * **Target Outcome:** `OFFENSE` (9 raw MPD offense categories mapped into a binary classification target: `Violent` vs. `Property` crime).
  * **Geographic Predictors:** `WARD`, `LATITUDE`, `LONGITUDE`.
  * **Temporal Predictors:** `SHIFT`, `START_DATE` (parsed into separate date and hour-of-day variables).
  * **Tactical Predictor / Descriptive Variable:** `METHOD` (`GUN`, `KNIFE`, `OTHERS`).

---
## 2. Data Loading and Initial Inspection (A01 Principles)

Following the course data wrangling rules, we never edit or overwrite the original source CSV files. We first load the raw data, check if both 2024 and 2025 files are present in the working directory to combine them cleanly, and inspect `.shape`, `.dtypes`, and `.head()` to see how the variables are recorded.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# load 2025 data and merge 2024 data if present in the local directory
file_2025 = 'data/Crime_Incidents_in_2025.csv'
file_2024 = 'data/Crime_Incidents_in_2024.csv'

df_2025 = pd.read_csv(file_2025, low_memory=False)
df_2025['source_year'] = 2025

if os.path.exists(file_2024):
    df_2024 = pd.read_csv(file_2024, low_memory=False)
    df_2024['source_year'] = 2024
    df = pd.concat([df_2024, df_2025], axis=0, ignore_index=True)
    print(f'Combined 2024 and 2025 datasets. Total shape: {df.shape}')
else:
    df = df_2025.copy()
    print(f'Loaded 2025 dataset (2024 CSV not yet in folder). Total shape: {df.shape}')

print('\nColumn Data Types:')
print(df.dtypes)

display(df.head())

Loaded 2025 dataset (2024 CSV not yet in folder). Total shape: (24152, 26)

Column Data Types:
X                       float64
Y                       float64
CCN                       int64
REPORT_DAT                  str
START_DATE                  str
END_DATE                    str
BLOCK                       str
OFFENSE                     str
METHOD                      str
SHIFT                       str
WARD                      int64
ANC                         str
DISTRICT                float64
PSA                     float64
NEIGHBORHOOD_CLUSTER        str
BLOCK_GROUP                 str
CENSUS_TRACT            float64
VOTING_PRECINCT             str
BID                         str
XBLOCK                  float64
YBLOCK                  float64
LATITUDE                float64
LONGITUDE               float64
OBJECTID                  int64
OCTO_RECORD_ID          float64
source_year               int64
dtype: object


,X,Y,CCN,REPORT_DAT,START_DATE,END_DATE,BLOCK,OFFENSE,METHOD,SHIFT,...,CENSUS_TRACT,VOTING_PRECINCT,BID,XBLOCK,YBLOCK,LATITUDE,LONGITUDE,OBJECTID,OCTO_RECORD_ID,source_year
0,400588.4650,129470.0424,5131706,2025/02/05 05:00:00+00,2005/09/27 07:25:00+00,2005/09/27 07:26:00+00,800 - 899 BLOCK OF XENIA STREET SE,HOMICIDE,GUN,MIDNIGHT,...,9801.0,Precinct 121,NaN,400588.464990,129470.042358,38.833020,-76.993222,1020985024,NaN,2025
1,405684.5761,135633.8979,10133557,2025/06/30 04:00:00+00,2010/09/15 00:45:00+00,2010/09/14 04:00:00+00,4600 - 4699 BLOCK OF A STREET SE,HOMICIDE,GUN,MIDNIGHT,...,9906.0,Precinct 104,NaN,405684.576067,135633.897851,38.888528,-76.934478,1020985027,NaN,2025
2,399950.1386,130512.4788,17194618,2025/06/30 04:00:00+00,2017/11/09 20:44:00+00,2017/11/09 22:57:34+00,3200 - 3299 BLOCK OF 4TH STREET SE,HOMICIDE,GUN,MIDNIGHT,...,9804.0,Precinct 122,NaN,399950.138558,130512.478775,38.842411,-77.000574,1020985039,NaN,2025
3,406954.0500,136726.6400,25175401,2025/11/18 22:51:31+00,2025/11/18 21:41:00+00,NaN,5600 - 5699 BLOCK OF NANNIE HELEN BURROUGHS AV...,ROBBERY,GUN,EVENING,...,7807.0,Precinct 95,NaN,406954.050000,136726.640000,38.898363,-76.919834,1021280459,NaN,2025
4,402158.3100,138824.5300,25175763,2025/11/19 18:07:47+00,2025/11/19 16:54:00+00,2025/11/19 17:00:00+00,1800 - 2299 BLOCK OF NEW YORK AVENUE NE,THEFT/OTHER,OTHERS,DAY,...,11100.0,Precinct 72,NaN,402158.310000,138824.530000,38.917286,-76.975113,1021280460,NaN,2025


---
## 3. Data Wrangling & Feature Construction

### 3.1 Fix Raw Variables and Missingness Data
Before building our models, we are going to clean up the missing data according to these rules:
1. Keeping all original columns intact.
2. For numeric predictors (`WARD`, `LATITUDE`, `LONGITUDE`), we use `pd.to_numeric(..., errors='coerce')` to create `_clean` columns and explicitly record missing or coerced values in `_nan` indicator columns.
3. We inspect `.unique()` and `.value_counts(dropna=False)` for `OFFENSE`, `SHIFT`, and `METHOD`, creating `_clean` and `_nan` columns for each.

In [38]:
# cleaning numeric geographic variables without silent overwrites
num_vars = ['WARD', 'LATITUDE', 'LONGITUDE']

for var in num_vars:
    df[f'{var}_clean'] = pd.to_numeric(df[var], errors='coerce')
    df[f'{var}_nan'] = df[f'{var}_clean'].isnull()

print('Missing or coerced values in numeric geographic variables:')
print(df[[f'{var}_nan' for var in num_vars]].sum(), '\n')

print('Summary statistics of cleaned numeric variables:')
display(df[[f'{var}_clean' for var in num_vars]].describe().round(4))

Missing or coerced values in numeric geographic variables:
WARD_nan         0
LATITUDE_nan     0
LONGITUDE_nan    0
dtype: int64 

Summary statistics of cleaned numeric variables:


,WARD_clean,LATITUDE_clean,LONGITUDE_clean
count,24152.0000,24152.0000,24152.0000
mean,4.3783,38.9095,-77.0097
std,2.2801,0.0313,0.0349
min,1.0000,38.8120,-77.1119
25%,2.0000,38.8960,-77.0322
50%,5.0000,38.9086,-77.0136
75%,6.0000,38.9276,-76.9886
max,8.0000,38.9942,-76.9100


In [39]:
# checking categorical variables OFFENSE, SHIFT, and METHOD
cat_vars = ['OFFENSE', 'SHIFT', 'METHOD']

for var in cat_vars:
    df[f'{var}_clean'] = df[var].astype('string').str.strip()
    df[f'{var}_nan'] = df[f'{var}_clean'].isnull() | (df[f'{var}_clean'] == '')
    print(f'{var} Value Counts:')
    print(df[f'{var}_clean'].value_counts(dropna=False), '\n')

OFFENSE Value Counts:
OFFENSE_clean
THEFT/OTHER                   11642
THEFT F/AUTO                   5362
MOTOR VEHICLE THEFT            3933
ROBBERY                        1323
ASSAULT W/DANGEROUS WEAPON      933
BURGLARY                        723
HOMICIDE                        127
SEX ABUSE                       105
ARSON                             4
Name: count, dtype: Int64 

SHIFT Value Counts:
SHIFT_clean
EVENING     10058
DAY          9951
MIDNIGHT     4143
Name: count, dtype: Int64 

METHOD Value Counts:
METHOD_clean
OTHERS    22463
GUN        1295
KNIFE       394
Name: count, dtype: Int64 



### 3.2 Target Construction: Dividing the 9 Crime Types into Violent vs. Property
Following our project proposal, we map the 9 official MPD `OFFENSE` types into two distinct categories:
* **Violent Crime (`is_violent = 1`):** `HOMICIDE`, `SEX ABUSE`, `ASSAULT W/DANGEROUS WEAPON`, `ROBBERY`
* **Property Crime (`is_violent = 0`):** `BURGLARY`, `THEFT F/AUTO`, `THEFT/OTHER`, `MOTOR VEHICLE THEFT`, `ARSON`

In [40]:
violent_offenses = [
    'HOMICIDE',
    'SEX ABUSE',
    'ASSAULT W/DANGEROUS WEAPON',
    'ROBBERY'
]

property_offenses = [
    'BURGLARY',
    'THEFT F/AUTO',
    'THEFT/OTHER',
    'MOTOR VEHICLE THEFT',
    'ARSON'
]

# construct categorical label and binary indicator in new columns
df['crime_category_clean'] = np.where(
    df['OFFENSE_clean'].isin(violent_offenses), 'Violent',
    np.where(df['OFFENSE_clean'].isin(property_offenses), 'Property', pd.NA)
)
df['crime_category_nan'] = df['crime_category_clean'].isnull()

df['is_violent_clean'] = np.where(
    df['crime_category_clean'] == 'Violent', 1.0,
    np.where(df['crime_category_clean'] == 'Property', 0.0, np.nan)
)

print('Crime Category Distribution:')
print(df['crime_category_clean'].value_counts(dropna=False))
print('\nProportions:')
print(df['crime_category_clean'].value_counts(normalize=True).round(4))

Crime Category Distribution:
crime_category_clean
Property    21664
Violent      2488
Name: count, dtype: int64

Proportions:
crime_category_clean
Property    0.897
Violent     0.103
Name: proportion, dtype: float64


### 3.3 Temporal Wrangling: Separating Date and Time Features from `START_DATE`
Continuing on based on our pitch deck, we parse `START_DATE` to extract separate date, hour of day (`0`–`23`), day of week, and month columns so we can analyze when each type of crime was committed. Because raw incident logs occasionally include historical reporting artifacts or unparseable timestamps, we track missing timestamps with `START_DATE_nan`.

In [41]:
# parse START_DATE into datetime format safely
df['START_DATE_dt'] = pd.to_datetime(df['START_DATE'], errors='coerce')
df['START_DATE_nan'] = df['START_DATE_dt'].isnull()

# extract separate date and time components
df['incident_date_clean'] = df['START_DATE_dt'].dt.date
df['hour_clean'] = df['START_DATE_dt'].dt.hour
df['month_clean'] = df['START_DATE_dt'].dt.month
df['dayofweek_clean'] = df['START_DATE_dt'].dt.day_name()

print(f"Missing/unparseable START_DATE values: {df['START_DATE_nan'].sum()}")
print('\nSummary of extracted hour_clean (0-23):')
print(df['hour_clean'].describe().round(2))

Missing/unparseable START_DATE values: 0

Summary of extracted hour_clean (0-23):
count    24152.00
mean        13.15
std          7.27
min          0.00
25%          6.00
50%         15.00
75%         19.00
max         23.00
Name: hour_clean, dtype: float64


### 3.4 Filtering to the Analytic Sample and Saving Clean Data
Now we are filtering out observations missing offense classifications or valid geographic coordinates (`LATITUDE_clean`, `LONGITUDE_clean`), as well as rows missing `hour_clean` or `WARD_clean`. We are then saved this all into a cleaned separate CSV file so the original raw file remains untouched.

In [ ]:
# Filter rows based on explicit exclusion criteria
valid_condition = (
    (~df['crime_category_nan']) &
    (~df['LATITUDE_nan']) &
    (~df['LONGITUDE_nan']) &
    (~df['WARD_nan']) &
    (~df['START_DATE_nan'])
)

df_clean = df.loc[valid_condition].copy()
print(f'Original row count (N): {len(df)}')
print(f'Cleaned analytic sample row count (N): {len(df_clean)}')
print(f'Rows excluded due to missingness: {len(df) - len(df_clean)}')

# Save a separate cleaned copy without overwriting the original CSV
df_clean.to_csv('data/Crime_Incidents_Clean+Combined.csv', index=False)

Original row count (N): 24152
Cleaned analytic sample row count (N): 24152
Rows excluded due to missingness: 0
